In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import os, sys, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.notebook import tqdm

sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/spikeparam')
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/')

from config import SPE1_PICKLE_ROOT, CELL_IDS, DICT_CELL_TYPE
from spikeparam.patch.fit import Spike

plt.rcParams['font.family'] = 'Helvetica Neue'

SPE1_PKL = SPE1_PICKLE_ROOT
PVC6_PKL = '/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/pvc6_pickles'

# Colours
COL_PC   = '#4878D0'   # spe-1 PC
COL_IN   = '#EE854A'   # spe-1 IN
COL_PVC6 = '#FF44CC'   # pvc-6 (both cells)

In [ ]:
FORCE_RERUN = False   # set True to refit pvc-6 and overwrite R² caches

## pvc-6 R² — refit from all_data pickles and cache

In [ ]:
def get_pvc6_r2(all_data_pkl, cache_pkl, fs=200000, force=False):
    """Fit Spike on pvc-6 all_data and cache r_squared_exp / r_squared_ramp."""
    if not force and os.path.exists(cache_pkl):
        with open(cache_pkl, 'rb') as f:
            return pickle.load(f)

    with open(all_data_pkl, 'rb') as f:
        all_data = pickle.load(f)
    all_data_flat = [s for sweep in all_data for s in sweep]

    sp = Spike(thresh_amp=-10, window_length=(5., 5.), smooth_frac=.008,
               pre_inflection_ms=0.5)
    sp.fit(all_data_flat, fs, n_jobs=-1, progress=tqdm)
    sp.gen_fit(ramp=True, exp=True)

    result = {
        'r_squared_exp':  np.asarray(sp.r_squared_exp,  dtype=float),
        'r_squared_ramp': np.asarray(sp.r_squared_ramp, dtype=float),
    }
    with open(cache_pkl, 'wb') as f:
        pickle.dump(result, f)
    print(f'Cached to {os.path.basename(cache_pkl)}')
    return result


pvc6_r2 = {
    'c1': get_pvc6_r2(
        os.path.join(PVC6_PKL, 'all_data.pkl'),
        os.path.join(PVC6_PKL, '_r2_c1.pkl'),
        force=FORCE_RERUN,
    ),
    'c2': get_pvc6_r2(
        os.path.join(PVC6_PKL, 'all_data2.pkl'),
        os.path.join(PVC6_PKL, '_r2_c2.pkl'),
        force=FORCE_RERUN,
    ),
}

for cid, d in pvc6_r2.items():
    print(f'pvc-6 {cid}: n={len(d["r_squared_exp"])}  '
          f'median r²_exp={np.nanmedian(d["r_squared_exp"]):.3f}  '
          f'median r²_ramp={np.nanmedian(d["r_squared_ramp"]):.3f}')

## spe-1 R² — load from spike_fit_pickles

In [ ]:
_fit_dir = os.path.join(SPE1_PKL, 'spike_fit_pickles')

spe1_r2 = []   # list of dicts: cell_id, cell_type, r2_exp, r2_ramp

for _cid in CELL_IDS:
    _p = os.path.join(_fit_dir, f'{_cid}_spike_fit.pkl')
    if not os.path.exists(_p):
        continue
    with open(_p, 'rb') as _f:
        _sp = pickle.load(_f)

    if _sp.r_squared_exp is None:
        _sp.gen_fit(ramp=True, exp=True)

    _cnum  = int(_cid.replace('c', ''))
    _ctype = DICT_CELL_TYPE.get(_cnum, 'PC')
    spe1_r2.append({
        'cell_id':   _cid,
        'cell_type': _ctype,
        'r2_exp':    np.asarray(_sp.r_squared_exp,  dtype=float),
        'r2_ramp':   np.asarray(_sp.r_squared_ramp, dtype=float),
    })

print(f'Loaded {len(spe1_r2)} spe-1 cells')
for d in spe1_r2:
    print(f"  {d['cell_id']} ({d['cell_type']}): n={len(d['r2_exp'])}  "
          f"med r²_exp={np.nanmedian(d['r2_exp']):.3f}  "
          f"med r²_ramp={np.nanmedian(d['r2_ramp']):.3f}")

## Plot options — violin / box / histogram

In [ ]:
_FS_AX  = 13
_FS_TK  = 9
_LW_SP  = 1.8

def plot_r2_panel(ax, r2_key, title):
    _x = 0
    xtick_pos  = []
    xtick_labs = []

    # ── spe-1 cells ───────────────────────────────────────────────────────────
    for d in spe1_r2:
        vals = d[r2_key]
        vals = vals[~np.isnan(vals)]
        col  = COL_IN if d['cell_type'] == 'IN' else COL_PC
        if len(vals) > 3:
            vp = ax.violinplot(vals, positions=[_x], widths=0.7,
                               showmedians=True, showextrema=False)
            for part in vp['bodies']:
                part.set_facecolor(col)
                part.set_alpha(0.65)
                part.set_edgecolor('none')
            vp['cmedians'].set_color('white')
            vp['cmedians'].set_linewidth(1.8)
        xtick_pos.append(_x)
        xtick_labs.append(d['cell_id'])
        _x += 1

    # separator
    ax.axvline(_x - 0.5, color='#aaaaaa', lw=1.0, ls='--')

    # ── pvc-6 cells ───────────────────────────────────────────────────────────
    for cid, d in pvc6_r2.items():
        vals = d[r2_key]
        vals = vals[~np.isnan(vals)]
        if len(vals) > 3:
            vp = ax.violinplot(vals, positions=[_x], widths=0.7,
                               showmedians=True, showextrema=False)
            for part in vp['bodies']:
                part.set_facecolor(COL_PVC6)
                part.set_alpha(0.75)
                part.set_edgecolor('none')
            vp['cmedians'].set_color('white')
            vp['cmedians'].set_linewidth(1.8)
        xtick_pos.append(_x)
        xtick_labs.append(f'pvc6\n{cid}')
        _x += 1

    ax.axhline(0.5, color='k', ls='--', lw=1.0, alpha=0.4)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xticks(xtick_pos)
    ax.set_xticklabels(xtick_labs, rotation=90, fontsize=_FS_TK - 1, fontweight='bold')
    ax.set_ylabel('R²', fontsize=_FS_AX, fontweight='bold')
    ax.set_title(title, fontsize=_FS_AX, fontweight='bold', pad=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for sp_ in ['bottom', 'left']:
        ax.spines[sp_].set_linewidth(_LW_SP)
    ax.tick_params(labelsize=_FS_TK)


fig, (ax_exp, ax_ramp) = plt.subplots(2, 1, figsize=(18, 10), sharey=True)

plot_r2_panel(ax_exp,  'r2_exp',  'Exponential decay fit  R²')
plot_r2_panel(ax_ramp, 'r2_ramp', 'Ramp fit  R²')
ax_ramp.set_ylabel('R²', fontsize=_FS_AX, fontweight='bold')

# Legend
_leg = [
    mpatches.Patch(facecolor=COL_PC,   alpha=0.65, label='spe-1  PC'),
    mpatches.Patch(facecolor=COL_IN,   alpha=0.65, label='spe-1  IN'),
    mpatches.Patch(facecolor=COL_PVC6, alpha=0.75, label='pvc-6'),
]
fig.legend(handles=_leg, loc='upper right', frameon=False,
           fontsize=_FS_AX, bbox_to_anchor=(1.0, 1.0))

plt.tight_layout()
plt.savefig('supp_r2_distributions.pdf', bbox_inches='tight', dpi=300)
plt.savefig('supp_r2_distributions.png', bbox_inches='tight', dpi=300)
plt.show()
print('Saved.')

### Option 1 — Box plots (per cell)